# 03 — EDA: Albums

Exploratory data analysis of the album dataset. Uses `ydata-profiling` to generate a comprehensive statistical report on album-level features including ratings, tags, country, label, and type distributions.

**What it does:**
- Loads `final_album_df` from the pickled DataFrame
- Downcasts numeric columns to reduce memory usage
- Samples 20,000 rows and runs `ydata-profiling` to produce an interactive HTML profile report covering distributions, missing values, correlations, and outliers

**Inputs:** `data/pickles/final_album_df.pkl`

**Outputs:** None — display only

**Run after:** `02-parquet-to-dataframes.ipynb`

## Dependencies

Import the libraries needed for this notebook. `numpy` and `pandas` are standard data manipulation tools. `matplotlib` is imported to ensure a display backend is available for any inline plots. `ProfileReport` from `ydata_profiling` is the main tool used at the end of the notebook — it auto-generates an interactive HTML report covering distributions, missing values, correlations, and outliers for any DataFrame passed to it.

In [5]:
import numpy as np
import pandas as pd
import matplotlib
from ydata_profiling import ProfileReport

## Load Album Data

Load `final_album_df` from disk. This pickle was produced by `02-parquet-to-dataframes.ipynb`, which joined MusicBrainz album, tag, and rating tables into a single denormalised DataFrame. Each row represents one album–tag pair, so albums with multiple tags appear multiple times. The shape printout confirms how many rows and columns were loaded — useful for catching truncated or corrupt pickle files before going further.

In [ ]:
# Load the master dataframe from the pickle we created
album_df = pd.read_pickle('../data/pickles/final_album_df.pkl')

print(f"Dataset loaded: {album_df.shape[0]:,} rows and {album_df.shape[1]} columns.")

## Optimise Memory Usage

Downcast numeric columns to their smallest viable dtype. When pandas reads a pickle it defaults to `float64` and `int64` for all numeric columns, regardless of the actual value range. `optimize_floats` scans every `float64` column and lets `pd.to_numeric(downcast='float')` shrink it to `float32` or smaller if no precision is lost. `optimize_ints` does the same for `int64`, potentially shrinking to `int32`, `int16`, or `int8`. On a dataset this size the saving is typically 40–60% of RAM, which matters when `ydata-profiling` later holds multiple copies of the data internally during analysis.

In [7]:
def optimize_floats(df):
    floats = df.select_dtypes(include=['float64']).columns
    df[floats] = df[floats].apply(pd.to_numeric, downcast='float')
    return df

def optimize_ints(df):
    ints = df.select_dtypes(include=['int64']).columns
    df[ints] = df[ints].apply(pd.to_numeric, downcast='integer')
    return df

album_df = optimize_floats(album_df)
album_df = optimize_ints(album_df)

## Inspect the First Rows

Preview the first five rows of `album_df` to confirm the schema looks correct after loading and dtype optimisation. Key things to check: `id` is an integer album identifier, `gid` is a raw MusicBrainz UUID stored as bytes (it is not decoded here — that happens upstream), `name` is the album title, `type` is an integer code for release type (e.g. 1 = Album), `tag_id` is the MusicBrainz tag identifier for this row's tag, `tag_count` is the community vote weight for that tag, and `rating` / `rating_count` are the aggregated user rating and number of raters. Repeated `id` values in these first rows indicate albums that have more than one tag.

In [8]:
album_df.head()

,id,gid,name,artist_credit,type,tag_id,tag_count,rating,rating_count
0,2,b'\xe8\xbe\xe7Y\x9e\xfc5\xc2\x93\xd7\t\xac\xe9...,Eclectic Electric,1,1,1186,2,0.0,0.0
1,2,b'\xe8\xbe\xe7Y\x9e\xfc5\xc2\x93\xd7\t\xac\xe9...,Eclectic Electric,1,1,92310,1,0.0,0.0
2,4,b'\x8bo\x13:/\xdf<\xc2\xb8M\x1c\x88\x9a\xdc\t9',Blue Lines,4,1,559,2,79.0,19.0
3,4,b'\x8bo\x13:/\xdf<\xc2\xb8M\x1c\x88\x9a\xdc\t9',Blue Lines,4,1,12,9,79.0,19.0
4,4,b'\x8bo\x13:/\xdf<\xc2\xb8M\x1c\x88\x9a\xdc\t9',Blue Lines,4,1,1498,14,79.0,19.0


## Profile Report

Sample 20,000 rows at random (seeded for reproducibility) and pass them to `ydata-profiling`. Sampling is necessary because running a full profile on the entire dataset would be prohibitively slow — 20,000 rows is large enough to give statistically reliable distributions while keeping report generation under a minute.

The `ProfileReport` with `minimal=False` (full mode) computes and renders:
- **Overview** — row/column counts, missing value totals, duplicate row counts, and memory usage after dtype optimisation
- **Per-column statistics** — for numeric columns: min, max, mean, median, std, kurtosis, skewness, and a histogram; for categorical columns: unique value counts and a frequency bar chart
- **Missing values heatmap** — reveals whether nulls in `rating` and `rating_count` are correlated (albums with no ratings tend to have no rating counts either)
- **Correlations** — Pearson and Spearman matrices; expect `rating` and `rating_count` to be moderately correlated since popular albums attract more votes
- **Interactions** — scatter plots between numeric pairs; useful for spotting non-linear relationships between `tag_count` and `rating`

The report renders inline as an iframe — scroll through it to identify columns that need cleaning or imputation before feature engineering.

In [10]:
# Take a 10-20% random sample
sample_df = album_df.sample(n=20000, random_state=42)

profile = ProfileReport(sample_df, title="Music Album Sample Profile", minimal=False)
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 9/9 [00:00<00:00, 89.28it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]